# 🔧 題目 2：B2C 電商競品比價
# Mini Data Pipeline 工作坊

> **情境**：你是電商平台的資料顧問。老闆想知道同一本書在 Amazon 和 Flipkart 價差有多大。
>
> **Pipeline**：`CSV → pandas → SQLite (raw/cleaned/analyzed) → SQL → LLM → FastAPI`
>
> **資料**：[Kaggle: Amazon vs Flipkart Book Prices](https://www.kaggle.com/datasets/mandan/amazon-vs-flipkart-book-prices)（2,000 筆取樣）
>
> 📄 詳細需求見 `requirements_spec.md`

---

### 📋 今日目標

| 必做（Section 1-8） | 回家作業（Section 9-10） |
|------|------|
| ✅ ETL pipeline（CSV → SQLite 三表） | 🏠 Dashboard（回家作業） |
| ✅ LLM 分析 + Pipeline Documentation | |
| ✅ LLM 分類分析 | ⭐ 本地部署 |
| ✅ Pipeline Documentation | |

### 🗺️ 標記說明

| 標記 | 意思 |
|------|------|
| `🟢 簡單` | 開放式 — 提示裡有範例教語法，自己應用 |
| `🟡 中等` | 半骨架 — 用別的情境示範，你翻譯到自己的欄位 |
| `🔴 較難` | 完整骨架 — 結構都給了，填關鍵處 |
| `（不需要改）` | 直接跑 |


## Section 0：環境設定

直接跑，不需要改。


In [ ]:
# （不需要改）Colab 環境自動設定
import os
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('topic_2'):
        !git clone https://github.com/lu791019/midterm-mvp-template.git /content/repo
    os.chdir('/content/repo/data/raw/topic_2')
    print("✅ Colab：已設定工作目錄 =", os.getcwd())
else:
    print("✅ 本地環境，工作目錄 =", os.getcwd())


In [ ]:
# （不需要改）
import pandas as pd
import sqlite3
import os
import json
print('✅ 套件載入完成')


In [ ]:
# （不需要改）
OPENAI_API_KEY = ""
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]
print("✅ API Key" if OPENAI_API_KEY else "⚠️ fallback 模式")


---
## Section 1：Extract — 讀取資料 + 寫入 raw 表

> pipeline 第一步：**資料進入系統**。


### 🎯 完成這段後你應該有：
- `pipeline.db` 裡有一張 **raw 表**（原始資料全部灌入）
- 知道資料有幾筆、幾欄、什麼型別

> ⏱ 時間不夠？只做 Step 1-1 和 1-4（讀 CSV + 存入 raw 表），跳過探索。


### Step 1-1 🟢 簡單：讀取 CSV

> 💡 `pd.read_csv()` 把 CSV 讀成 DataFrame
> 💡 **範例**：如果要讀超市庫存資料：
> ```python
> df = pd.read_csv("inventory.csv")
> print(f"共 {len(df)} 筆")
> print(list(df.columns))
> df.head()
> ```
> 🎯 現在對 `products.csv` 做同樣的事
> ✅ 預期：2,000 筆


In [ ]:
# TODO 🟢: 讀取 products.csv

# 相關程式碼：
# df_raw = pd.read_csv("products.csv")
# print(f"📊 {len(df_raw)} 筆, {len(df_raw.columns)} 欄")
# print(list(df_raw.columns))
# df_raw.head()


### Step 1-2 🟢 簡單：檢查資料品質

> 💡 **範例**：檢查超市庫存：
> ```python
> print(df.dtypes)
> print(df.isnull().sum())
> print(df.describe())
> ```
> 🎯 對 df_raw 做同樣三件事


In [ ]:
# TODO 🟢: 檢查品質

# 相關程式碼：
# print(df_raw.dtypes)
# print(df_raw.isnull().sum())
# print(df_raw.describe())


### Step 1-3 🟢 簡單：自由探索

> 💡 **範例**：
> ```python
> df["product_name"].value_counts().head(10)
> df["product_name"].nunique()
> df.sample(5)
> ```
> 🎯 用上面的招式探索你的資料


In [ ]:
# TODO 🟢: 自由探索

# 相關程式碼：
# df_raw["isbn"].value_counts().head(10)
# df_raw["isbn"].nunique()


### Step 1-4 🟡 中等：建立 SQLite + 寫入 raw 表

> 💡 **範例**：把超市庫存存進資料庫：
> ```python
> conn = sqlite3.connect("warehouse.db")
> df.to_sql("raw_inventory", conn, if_exists="replace", index=False)
> result = pd.read_sql("SELECT COUNT(*) as total FROM raw_inventory", conn)
> print(f"raw_inventory: {result['total'][0]} 筆")
> ```
> 🎯 建立 `pipeline.db`，寫入 `raw_products` 表
> ✅ 預期：raw_products: 2000 筆


In [ ]:
# TODO 🟡: 建立 SQLite，寫入 raw 表
DB_PATH = "pipeline.db"

# 相關程式碼：
# conn = sqlite3.connect(DB_PATH)
# df_raw.to_sql("raw_products", conn, if_exists="replace", index=False)
# ...驗證筆數


---
## Section 2：Transform — 清洗 + 寫入 cleaned 表

> 從**資料庫**讀出 → 清洗 → 寫回資料庫。


### 🎯 完成這段後你應該有：
- `pipeline.db` 裡有一張 **cleaned 表**（已清洗）
- 清洗邏輯包含：去缺值、轉型別、必要時新增欄位

> ⏱ 時間不夠？只做 Step 2-1、2-2 和 2-5（讀出 + 基本清洗 + 存入 cleaned 表），跳過進階處理。


### Step 2-1 🟢 簡單：從 raw 表讀出

> 💡 **範例**：`pd.read_sql("SELECT * FROM raw_inventory", conn)`
> ✅ 預期：df 有 2000 筆


In [ ]:
# TODO 🟢: 從 raw_products 讀出


### Step 2-2 🟢 簡單：處理缺漏值

> 💡 **範例**：`df.dropna(subset=["product_name", "price"])`
> 🎯 刪除 `title` 和 `amazon_price` 為空的列


In [ ]:
# TODO 🟢: 刪除缺漏值


### Step 2-3 🟡 中等：把價格欄位轉數字、計算價差、標記哪個平台便宜

> 💡 **範例**：假設有超市庫存，要比較 A 店和 B 店的價差：
> ```python
> inv["store_a_price"] = pd.to_numeric(inv["store_a_price"], errors="coerce")
> inv["store_b_price"] = pd.to_numeric(inv["store_b_price"], errors="coerce")
> inv["price_diff"] = inv["store_a_price"] - inv["store_b_price"]
> inv["cheaper"] = inv.apply(lambda r: "A店" if r["price_diff"] < 0 else "B店", axis=1)
> ```
> 🎯 對你的資料：把 amazon_price / flipkart_price 轉數字，計算 price_diff 和 cheaper_on


In [ ]:
# TODO 🟡: 清洗轉換


### 🏁 清洗檢查點

> 直接跑。全部 ✅ 才往下。


In [ ]:
# （不需要改）
print(f"清洗前→清洗後: {len(df)} 筆")
print(f"缺漏值: {df.isnull().sum().sum()}")
assert df.isnull().sum().sum() == 0, "❌ 還有缺漏值"
print("✅ 檢查通過")


### Step 2-4 🟢 簡單：寫入 cleaned 表

> 💡 跟 Step 1-4 一樣用 `to_sql`：
> ```python
> df_clean.to_sql("cleaned_products", conn, if_exists="replace", index=False)
> print(f"✅ cleaned_products: {len(df_clean)} 筆")
> ```
> ✅ 預期：cleaned_products 筆數 ≤ raw 表


In [ ]:
# TODO 🟢: 寫入 cleaned_products 表


---
## Section 3：統計分析（pandas / SQL）

> 用 SQL 從資料庫查詢。


### 🎯 完成這段後你應該有：
- 至少一張統計表（`processed/*.csv`）
- 至少一個可以在 Demo 裡講的數字

> ⏱ 時間不夠？只做 Step 3-1（一個 GROUP BY 查詢），跳過視覺化和自由探索。


### Step 3-1 🟡 中等：價差最大的書 Top 20

> 💡 **範例**：查各部門平均考績：
> ```python
> pd.read_sql("""
>     SELECT department, COUNT(*), ROUND(AVG(score), 2)
>     FROM employees GROUP BY department ORDER BY AVG(score) DESC
> """, conn)
> ```
> 🎯 從 `cleaned_products` 查價差最大的書 Top 20
> 💡 提示：ORDER BY ABS(price_diff) DESC


In [ ]:
# TODO 🟡: 價差最大的書 Top 20
stat1 = pd.read_sql("""

""", conn)
stat1

# Skeleton:
# price_stats = pd.read_sql("""
#     SELECT title, amazon_price, flipkart_price,
#            (flipkart_price - amazon_price) AS price_diff
#     FROM cleaned_products ORDER BY ABS(price_diff) DESC LIMIT 20
# """, conn)



### Step 3-2 🟡 中等：各平台比較

> 💡 提示：GROUP BY cheaper_on，看哪個平台整體便宜


In [ ]:
# TODO 🟡: 價差分析 SQL

# Hint: 算 flipkart_price - amazon_price，用 ORDER BY ABS(price_diff)
# Skeleton:
# price_stats = pd.read_sql("""
#     SELECT title, amazon_price, flipkart_price,
#            (flipkart_price - amazon_price) AS price_diff
#     FROM cleaned_products ORDER BY ABS(price_diff) DESC LIMIT 20
# """, conn)

price_stats = pd.read_sql("""

""", conn)
price_stats


### Step 3-3 🟡 中等：視覺化

> 💡 **範例**：`df.plot.barh(x="department", y="avg_score", figsize=(10,5))`
> 🎯 把統計結果畫成至少一張圖


In [ ]:
# TODO 🟡: 視覺化
import matplotlib.pyplot as plt


### Step 3-4 🟢 簡單：自由探索 SQL

> 💡 靈感：平均評分差異？哪些作者的書最多？


In [ ]:
# TODO 🟢: 你自己的 SQL


### Step 3-5 🟢 簡單：存統計結果

> 💡 `os.makedirs("processed", exist_ok=True)` + `df.to_csv()`


In [ ]:
# TODO 🟢: 存結果到 processed/


### 💡 你還可以分析什麼？（進階探索）

- 💰 **價差分析**：哪些書 Amazon 比 Flipkart 便宜？反過來呢？
- ⭐ **評分差異**：同一本書兩平台評分差多少？
- 📈 **價格區間分佈**：大部分書落在什麼價位？
- 🏷️ **作者分析**：哪些作者的書平均最貴/最便宜？

> 試試 `CASE WHEN amazon_price > flipkart_price THEN 'Amazon貴' ELSE 'Flipkart貴' END`


---
## Section 4：LLM 加值分析

> 對書名做品類分類（科技/商業/文學/教育/生活/其他）。helper 函式已寫好，你要：呼叫、看結果、跑批次、寫入資料庫。


### 🎯 完成這段後你應該有：
- `pipeline.db` 裡有一張 **analyzed 表**（Bronze/Silver/Gold 三表齊全）
- LLM 或 fallback 分析結果寫入 analyzed 表

> ⏱ 沒有 API Key？直接用 fallback 規則版，一樣能完成。


In [ ]:
import requests
def llm_analyze(text, api_key=None):
    if api_key: return _llm_api(text, api_key)
    return _llm_fallback(text)
def _llm_api(text, api_key):
    prompt = f"""請分析以下書籍資訊，回傳 JSON：
{{"category": "文學/科技/商業/教育/生活/其他", "insight": "一句話書籍分析"}}\n文字：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3}, timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        if content.startswith("```"): content = content.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(content)
    except: return _llm_fallback(text)
def _llm_fallback(text):
    t = text.lower()
    if any(w in t for w in ["python","java","code","programming","data"]): cat = "科技"
    elif any(w in t for w in ["business","management","marketing","finance"]): cat = "商業"
    elif any(w in t for w in ["cook","health","fitness","travel","garden"]): cat = "生活"
    elif any(w in t for w in ["learn","study","guide","handbook","education"]): cat = "教育"
    elif any(w in t for w in ["novel","fiction","story","poetry","literature"]): cat = "文學"
    else: cat = "其他"
    return {"category": cat, "insight": text[:50] + "..."}
print("✅ LLM Helper")


# 批次分析 helper（不需要改）
def run_batch_analysis(df, text_column, conn, table_name, n=50, api_key=None):
    """一行搞定：批次 LLM 分析 + 寫入 analyzed 表。"""
    df_batch = df.head(n).copy()
    results = []
    for i, row in df_batch.iterrows():
        r = llm_analyze(str(row[text_column]), api_key)
        results.append(r)
        if len(results) % 10 == 0:
            print(f"  進度: {len(results)}/{n}")
    df_batch["category"] = [r.get("category", "") for r in results]
    df_batch["llm_insight"] = [r.get("insight", "") for r in results]
    df_batch.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"✅ {table_name}: {len(df_batch)} 筆已寫入")
    return df_batch



### Step 4-1 🟢 簡單：單筆測試

> 💡 **範例**：
> ```python
> test = df["product_name"].iloc[0]
> result = analyze(test)
> print(result)
> ```
> 🎯 取 df 的第一筆 `title`，呼叫 `llm_analyze()`


In [ ]:
# TODO 🟢: 單筆測試


### Step 4-2 🟢 簡單：批次分析 + 寫入 analyzed 表

> 💡 上面的 `run_batch_analysis()` 幫你一行搞定：批次呼叫 LLM + 整理結果 + 寫入資料庫
>
> 🎯 呼叫 `run_batch_analysis(df, "product_name", conn, "analyzed_products")` 
> ✅ 預期：`analyzed_products` 表有 50 筆，多了 `category` 和 `llm_insight` 欄位


In [ ]:
# TODO 🟢: 一行搞定批次 LLM 分析
df_analyzed = run_batch_analysis(df, "product_name", conn, "analyzed_products", n=50, api_key=OPENAI_API_KEY if OPENAI_API_KEY else None)
df_analyzed.head()


### ✅ 檢查點：三表驗證

> 跑完下面這格確認三張表都有資料。


In [ ]:
# TODO 🟡: 跨表查詢
lineage = pd.read_sql("""

""", conn)
print(lineage.to_string(index=False))


---
## Section 6：Pipeline Documentation


### 🎯 完成這段後你應該有：
- `output/pipeline_doc.md` 有 Pipeline Documentation（用 AI prompt 快速產出）

> ⏱ 時間不夠？先用 AI prompt 產草稿，課後再補完整版。
> 📋 課後把這份文件整理成你自己 repo 的 README.md。


### Step 6-1 🟢 簡單：寫報告

> 💡 用 f-string 嵌入數字，存到 output/pipeline_doc.md
> 🎯 報告要有具體數字和建議


In [ ]:
# TODO 🟢: 寫報告
report = f"""# B2C 電商競品比價分析報告

## 資料概要
（填入分析筆數、關鍵統計）

## 關鍵發現
（根據 Section 3 統計，寫 2-3 個發現）

## 建議
（寫 2-3 條有數據支撐的建議）

## Pipeline
CSV → pandas → SQLite → SQL → LLM → 本報告
"""
os.makedirs("output", exist_ok=True)
with open("output/pipeline_doc.md", "w") as f: f.write(report)
print("✅ pipeline_doc.md")


---
## Section 7：Next Step 規劃

> 已包含在 `pipeline_doc.md` 的 Section 10「Known Limitations & Next Steps」。
> 回到 Section 6 產出的文件，把升級計畫填在最後一區即可。


In [ ]:
# （不需要改）
checks = [("pipeline.db","DB"), ("processed","統計"), ("output/pipeline_doc.md","Pipeline Doc")]
for p,d in checks: print(f"  {'✅' if os.path.exists(p) else '❌'} {d}: {p}")
if os.path.exists("pipeline.db"):
    c = sqlite3.connect("pipeline.db")
    for t in ["raw_products","cleaned_products","analyzed_products"]:
        try: print(f"  ✅ {t}: {pd.read_sql(f'SELECT COUNT(*) as n FROM [{t}]', c)['n'][0]}")
        except: print(f"  ❌ {t}")
    c.close()
print("\n📋 接下來：pipeline_doc.md 補完 + Demo 準備 + Section 9-10（回家作業）")


---
## Section 8：FastAPI

> 🅰️ 在下面寫 / 🅱️ 開 `api.py`（solution）


### 🎯 完成這段後你應該有：
- `api.py` 能跑起來
- 瀏覽器打開 `http://localhost:8000/health` 回 200

> ⏱ 卡住了？`api.py` 已經是 solution，改好路徑直接跑即可。


### Step 8-1 🔴 較難：定義 endpoint

> 💡 **範例**：把考績做成 API：
> ```python
> from fastapi import FastAPI
> api = FastAPI(title="考績 API")
> @api.get("/top")
> def top():
>     c = sqlite3.connect("school.db")
>     df = pd.read_sql("SELECT name, score FROM employees ORDER BY score DESC LIMIT 10", c)
>     c.close()
>     return df.to_dict(orient="records")
> ```
> 🎯 定義 /health、/stats（價差排行）、/analyzed


In [ ]:
# TODO 🔴: FastAPI
!pip install -q fastapi uvicorn nest_asyncio
from fastapi import FastAPI
import nest_asyncio
nest_asyncio.apply()

api = FastAPI(title="B2C 電商競品比價 API")

@api.get("/health")
def health():
    return {"status": "ok"}

# TODO: /stats


# TODO: /analyzed


print("✅ API 定義完成")


### Step 8-2 🟡 中等：啟動 + 測試


In [ ]:
# 啟動（直接跑）
import threading, uvicorn, time
thread = threading.Thread(target=uvicorn.run, args=(api,), kwargs={"host":"0.0.0.0","port":8000,"log_level":"warning"})
thread.daemon = True
thread.start()
time.sleep(2)
print("✅ API 已啟動")

# TODO 🟡: 用 requests 測試
import requests


> 🅱️ `api.py` 是 solution。本地：`uvicorn api:app --reload --port 8000`


---
## Section 9（回家作業）：Dashboard

> 🅰️ ipywidgets（Colab）/ 🅱️ `app.py`（Streamlit，solution）


### Step 9-1 🔴 較難：互動 Dashboard

> 💡 **範例**：選部門看薪資分佈：
> ```python
> import ipywidgets as widgets
> from IPython.display import display, clear_output
> dept_dd = widgets.Dropdown(options=["全部","工程部","業務部"], description="部門：")
> def update(dept):
>     clear_output(wait=True)
>     display(dept_dd)
>     data = df if dept == "全部" else df[df["department"] == dept]
>     print(f"{dept}: {len(data)} 人")
>     data["salary"].hist()
>     plt.show()
> widgets.interact(update, dept=dept_dd)
> ```
> 🎯 做一個「選 cheaper_on → 看 price_diff 分佈」的互動


In [ ]:
# TODO 🟡: 價差分析 SQL

# Hint: 算 flipkart_price - amazon_price，用 ORDER BY ABS(price_diff)
# Skeleton:
# price_stats = pd.read_sql("""
#     SELECT title, amazon_price, flipkart_price,
#            (flipkart_price - amazon_price) AS price_diff
#     FROM cleaned_products ORDER BY ABS(price_diff) DESC LIMIT 20
# """, conn)

price_stats = pd.read_sql("""

""", conn)
price_stats


> 🅱️ `app.py` 是 solution。本地：`streamlit run app.py`


---
## Section 10（回家作業）：本地部署指引

```bash
cd data/raw/topic_2
uvicorn api:app --reload --port 8000    # Terminal 1
streamlit run app.py                     # Terminal 2
```

| 現在 | 升級後 | 對應課程 |
|------|--------|---------|
| SQLite | MySQL / BigQuery | 資料庫模組 |
| 手動跑 | Airflow DAG | Airflow 模組 |
| 本地 Streamlit | Docker 容器化 | Docker 模組 |
| 本地開發 | GCP 雲端部署 | GCP 模組 |
